# Projeto 3

## João Victor De Bortoli Prado - 13672071

## Conceito

* Garagem na beira de estrada
* Objetos internos: lâmpada, pneu, ferramentas, gasolina
* Objetos externos: árvores, póstes, cones
* Objeto extra: carro
* Ambiente: estrada, grama, terra, garagem, portão

## Iluminação

* Ambiente externo: faróis do carro e lâmpada dos postes
* Ambiente interno: lâmpada de teto

## Controles
* ====================================
* W/A/S/D - movimentação da câmera
* P     - liga/desliga modo wireframe
* R     - reseta a cena
* ↓/↑   - movimenta carro para frete/trás
* ←/→   - move o volante (do carro) para esquerda/direita
* SPACEBAR - move o volante (do carro) para o centro
* I - enche o pneu
* K - esvazia o pneu
* O - abre o portão
* L - fecha o portão
* ====================================
* 1 - liga/desliga iluminação ambiente
* 2 - liga/desliga lâmpada de teto
* 3 - liga/desliga faróis
* 4 - liga/desliga lanterna
* 5 - liga/desliga postes
* Z/X - aumenta/diminui iluminação ambiente
* C/V - aumenta/diminui iluminação difusa
* B/N - aumenta/diminui iluminação especular
* ====================================


### Importando bibliotecas necessárias

In [379]:
import glfw
from OpenGL.GL import *
import numpy as np
import glm
import math
from numpy import random
from PIL import Image

from shader_s import Shader

### Inicializando janela

In [380]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)

altura = 700
largura = 700

window = glfw.create_window(largura, altura, "Programa", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)

### Constroi, compila e "linka" shaders aos programas

In [381]:
ourShader = Shader("vertex_shader.vs", "fragment_shader.fs")
ourShader.use()

program = ourShader.getProgram()

### Preparando dados e carregando modelos

In [382]:
# glEnable(GL_TEXTURE_2D)
glHint(GL_LINE_SMOOTH_HINT, GL_DONT_CARE)
glEnable( GL_BLEND )
glBlendFunc( GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA )
glEnable(GL_LINE_SMOOTH)
glEnable(GL_CULL_FACE)
glCullFace(GL_BACK)


global vertices_list
vertices_list = []    
global textures_coord_list
textures_coord_list = []
global normals_list
normals_list = []


def load_model_from_file(filename):
    """Loads a Wavefront OBJ file. """
    objects = {}
    vertices = []
    texture_coords = []
    normals = []
    faces = []

    material = None

    # abre o arquivo obj para leitura
    for line in open(filename, "r"): ## para cada linha do arquivo .obj
        if line.startswith('#'): continue ## ignora comentarios
        values = line.split() # quebra a linha por espaço
        if not values: continue

        ### recuperando vertices
        if values[0] == 'v':
            vertices.append(values[1:4])

        ### recuperando coordenadas de textura
        elif values[0] == 'vt':
            texture_coords.append(values[1:3])

        ### recuperando normais
        elif values[0] == 'vn':
            normals.append(values[1:4])

        ### recuperando faces 
        elif values[0] in ('usemtl', 'usemat'):
            if len(values) > 1:
                material = values[1]
            else:
                material = "default"
        elif values[0] == 'f':
            face = []
            face_texture = []
            face_normal = []
            for v in values[1:]:
                w = v.split('/')

                face.append(int(w[0]))

                if len(w) >= 2 and len(w[1]) > 0:
                    face_texture.append(int(w[1]))
                else:
                    face_texture.append(0)

                if len(w) >= 3 and len(w[2]) > 0:
                    face_normal.append(int(w[2]))
                else:
                    face_normal.append(0)

            faces.append((face, face_texture, face_normal, material))

    model = {}
    model['vertices'] = vertices
    model['texture'] = texture_coords
    model['normals'] = normals
    model['faces'] = faces

    return model


def load_texture_from_file(texture_id, img_textura):
    print(texture_id)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR_MIPMAP_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    img = Image.open(img_textura)
    img_width = img.size[0]
    img_height = img.size[1]
    image_data = img.tobytes("raw", "RGB", 0, -1)
    #image_data = np.array(list(img.getdata()), np.uint8)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, img_width, img_height, 0, GL_RGB, GL_UNSIGNED_BYTE, image_data)
    glGenerateMipmap(GL_TEXTURE_2D)
    try:
        from OpenGL.GL.EXT.texture_filter_anisotropic import GL_TEXTURE_MAX_ANISOTROPY_EXT
        glTexParameterf(GL_TEXTURE_2D, GL_TEXTURE_MAX_ANISOTROPY_EXT, 16.0)
    except:
        pass

def load_texture_from_file_skybox(texture_id, img_textura):
    print(texture_id)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_CLAMP_TO_EDGE)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_CLAMP_TO_EDGE)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR_MIPMAP_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    img = Image.open(img_textura)
    img_width = img.size[0]
    img_height = img.size[1]
    image_data = img.tobytes("raw", "RGB", 0, -1)
    #image_data = np.array(list(img.getdata()), np.uint8)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, img_width, img_height, 0, GL_RGB, GL_UNSIGNED_BYTE, image_data)
    glGenerateMipmap(GL_TEXTURE_2D)
    try:
        from OpenGL.GL.EXT.texture_filter_anisotropic import GL_TEXTURE_MAX_ANISOTROPY_EXT
        glTexParameterf(GL_TEXTURE_2D, GL_TEXTURE_MAX_ANISOTROPY_EXT, 16.0)
    except:
        pass



'''
É possível encontrar, na Internet, modelos .obj cujas faces não sejam triângulos. Nesses casos, precisamos gerar triângulos a partir dos vértices da face.
A função abaixo retorna a sequência de vértices que permite isso. Créditos: Hélio Nogueira Cardoso e Danielle Modesti (SCC0650 - 2024/2).
'''
def circular_sliding_window_of_three(arr):
    if len(arr) == 3:
        return arr
    circular_arr = arr + [arr[0]]
    result = []
    for i in range(len(circular_arr) - 2):
        result.extend(circular_arr[i:i+3])
    return result
    
global numberTextures
numberTextures = 0

def load_obj_and_texture(objFile, texturesList):
    modelo = load_model_from_file(objFile)
    
    ### inserindo vertices do modelo no vetor de vertices
    verticeInicial = len(vertices_list)
    print('Processando modelo {}. Vertice inicial: {}'.format(objFile, len(vertices_list)))
    # faces_visited = []
    for face in modelo['faces']:
        # if face[2] not in faces_visited:
            # faces_visited.append(face[2])
        for vertice_id in circular_sliding_window_of_three(face[0]):
            vertices_list.append(modelo['vertices'][vertice_id - 1])
        for texture_id in circular_sliding_window_of_three(face[1]):
            textures_coord_list.append(modelo['texture'][texture_id - 1])
        for normal_id in circular_sliding_window_of_three(face[2]):
            normals_list.append(modelo['normals'][normal_id - 1])
        
    verticeFinal = len(vertices_list)
    print('Processando modelo {}. Vertice final: {}'.format(objFile, len(vertices_list)))
    
    ### carregando textura equivalente e definindo um id (buffer): use um id por textura!
    global numberTextures
    for i in range(len(texturesList)):
        load_texture_from_file(numberTextures,texturesList[i])
        numberTextures += 1
    
    return verticeInicial, verticeFinal - verticeInicial

### Definindo funções de desenho para cada modelo

In [383]:
# ==========================================
# OBJETOS DO CENARIO
# ==========================================

# carrega rua (modelo e texturas)
verticeInicial_rua, quantosVertices_rua = load_obj_and_texture('objetos/rua/rua.obj', ['objetos/rua/rua.png'])

def desenha_rua(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
            
    # define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)

    # Parâmetros de iluminação alinhados como float com o Shader
    glUniform1f(glGetUniformLocation(program, "matAmbient"), 0.2)
    glUniform1f(glGetUniformLocation(program, "matDiffuse"), 0.5)
    glUniform1f(glGetUniformLocation(program, "matSpecular"), 0.1)
    glUniform1f(glGetUniformLocation(program, "matShininess"), 8.0)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_rua, quantosVertices_rua) ## renderizando


# carrega poste (modelo e texturas)
verticeInicial_poste, quantosVertices_poste = load_obj_and_texture('objetos/poste/poste.obj', ['objetos/poste/poste.png'])

def desenha_poste(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
            
    # define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)

    glUniform1f(glGetUniformLocation(program, "matAmbient"), 0.2)
    glUniform1f(glGetUniformLocation(program, "matDiffuse"), 0.6)
    glUniform1f(glGetUniformLocation(program, "matSpecular"), 0.3)
    glUniform1f(glGetUniformLocation(program, "matShininess"), 16.0)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_poste, quantosVertices_poste) ## renderizando


# carrega carro (modelo e texturas)
verticeInicial_carro, quantosVertices_carro = load_obj_and_texture('objetos/carro/carro.obj', ['objetos/carro/carro.png'])

def desenha_carro(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
            
    # define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)

    glUniform1f(glGetUniformLocation(program, "matAmbient"), 0.2)
    glUniform1f(glGetUniformLocation(program, "matDiffuse"), 1.0)
    glUniform1f(glGetUniformLocation(program, "matSpecular"), 1.0) # Brilho especular alto para a lataria
    glUniform1f(glGetUniformLocation(program, "matShininess"), 64.0)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_carro, quantosVertices_carro) ## renderizando


# carrega pneu (modelo e texturas)
verticeInicial_pneu, quantosVertices_pneu = load_obj_and_texture('objetos/pneu/pneu.obj', ['objetos/pneu/pneu.png'])

def desenha_pneu(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
            
    # define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)

    glUniform1f(glGetUniformLocation(program, "matAmbient"), 0.1)
    glUniform1f(glGetUniformLocation(program, "matDiffuse"), 0.2)
    glUniform1f(glGetUniformLocation(program, "matSpecular"), 0.05) # Borracha fosca reflete muito pouco
    glUniform1f(glGetUniformLocation(program, "matShininess"), 4.0)    
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_pneu, quantosVertices_pneu) ## renderizando


# carrega cone (modelo e texturas)
verticeInicial_cone, quantosVertices_cone = load_obj_and_texture('objetos/cone/cone.obj', ['objetos/cone/cone.png'])

def desenha_cone(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
            
    # define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # Aplicação dos parâmetros base (Plástico do cone)
    glUniform1f(glGetUniformLocation(program, "matAmbient"), 0.2)
    glUniform1f(glGetUniformLocation(program, "matDiffuse"), 0.8)
    glUniform1f(glGetUniformLocation(program, "matSpecular"), 0.2)
    glUniform1f(glGetUniformLocation(program, "matShininess"), 8.0)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_cone, quantosVertices_cone) ## renderizando


# carrega ferramentas (modelo e texturas)
verticeInicial_ferramentas, quantosVertices_ferramentas = load_obj_and_texture('objetos/ferramentas/ferramentas.obj', ['objetos/ferramentas/ferramentas.png'])

def desenha_ferramentas(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
            
    # define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # Aplicação dos parâmetros base (Metal refletivo das ferramentas)
    glUniform1f(glGetUniformLocation(program, "matAmbient"), 0.2)
    glUniform1f(glGetUniformLocation(program, "matDiffuse"), 0.6)
    glUniform1f(glGetUniformLocation(program, "matSpecular"), 0.7)
    glUniform1f(glGetUniformLocation(program, "matShininess"), 32.0)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_ferramentas, quantosVertices_ferramentas) ## renderizando


# carrega gasolina (modelo e texturas)
verticeInicial_gasolina, quantosVertices_gasolina = load_obj_and_texture('objetos/gasolina/gasolina.obj', ['objetos/gasolina/gasolina.png'])

def desenha_gasolina(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
            
    # define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # Aplicação dos parâmetros base (Plástico do galão)
    glUniform1f(glGetUniformLocation(program, "matAmbient"), 0.2)
    glUniform1f(glGetUniformLocation(program, "matDiffuse"), 0.5)
    glUniform1f(glGetUniformLocation(program, "matSpecular"), 0.15)
    glUniform1f(glGetUniformLocation(program, "matShininess"), 10.0)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_gasolina, quantosVertices_gasolina) ## renderizando


# carrega lampada (modelo e texturas)
verticeInicial_lampada, quantosVertices_lampada = load_obj_and_texture('objetos/lampada/lampada.obj', ['objetos/lampada/lampada.png'])

def desenha_lampada(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
            
    # define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # Ganho ambiente alto para o vidro da lâmpada parecer que emite luz própria
    glUniform1f(glGetUniformLocation(program, "matAmbient"), 0.8)
    glUniform1f(glGetUniformLocation(program, "matDiffuse"), 0.3)
    glUniform1f(glGetUniformLocation(program, "matSpecular"), 0.6)
    glUniform1f(glGetUniformLocation(program, "matShininess"), 32.0)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_lampada, quantosVertices_lampada) ## renderizando


# carrega arvore (modelo e texturas)
verticeInicial_arvore, quantosVertices_arvore = load_obj_and_texture('objetos/arvore/arvore.obj', ['objetos/arvore/arvore.png'])

def desenha_arvore(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
            
    # define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    glUniform1f(glGetUniformLocation(program, "matAmbient"), 0.3)
    glUniform1f(glGetUniformLocation(program, "matDiffuse"), 0.6)
    glUniform1f(glGetUniformLocation(program, "matSpecular"), 0.05) # Árvores são opacas
    glUniform1f(glGetUniformLocation(program, "matShininess"), 2.0)

    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_arvore, quantosVertices_arvore) ## renderizando


# carrega lanterna (modelo e texturas)
verticeInicial_lanterna, quantosVertices_lanterna = load_obj_and_texture('objetos/lanterna/lanterna.obj', ['objetos/lanterna/lanterna.png'])

def desenha_lanterna(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
            
    # define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # Aplicação dos parâmetros base (Carcaça metálica/plástica da lanterna)
    glUniform1f(glGetUniformLocation(program, "matAmbient"), 0.4)
    glUniform1f(glGetUniformLocation(program, "matDiffuse"), 0.6)
    glUniform1f(glGetUniformLocation(program, "matSpecular"), 0.4)
    glUniform1f(glGetUniformLocation(program, "matShininess"), 16.0)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_lanterna, quantosVertices_lanterna) ## renderizando

Processando modelo objetos/rua/rua.obj. Vertice inicial: 0
Processando modelo objetos/rua/rua.obj. Vertice final: 90720
0
Processando modelo objetos/poste/poste.obj. Vertice inicial: 90720
Processando modelo objetos/poste/poste.obj. Vertice final: 132366
1
Processando modelo objetos/carro/carro.obj. Vertice inicial: 132366
Processando modelo objetos/carro/carro.obj. Vertice final: 152601
2
Processando modelo objetos/pneu/pneu.obj. Vertice inicial: 152601
Processando modelo objetos/pneu/pneu.obj. Vertice final: 361641
3
Processando modelo objetos/cone/cone.obj. Vertice inicial: 361641
Processando modelo objetos/cone/cone.obj. Vertice final: 369009
4
Processando modelo objetos/ferramentas/ferramentas.obj. Vertice inicial: 369009
Processando modelo objetos/ferramentas/ferramentas.obj. Vertice final: 429639
5
Processando modelo objetos/gasolina/gasolina.obj. Vertice inicial: 429639
Processando modelo objetos/gasolina/gasolina.obj. Vertice final: 484563
6
Processando modelo objetos/lampada/

In [384]:
def cria_skybox(texturas):
    global vertices_list, textures_coord_list

    size = 200.0

    # vértices (36 = 6 faces * 2 triângulos * 3 vértices)
    cube = [
        # frente
        (-size, -size,  size), ( size, -size,  size), ( size,  size,  size),
        (-size, -size,  size), ( size,  size,  size), (-size,  size,  size),

        # direita
        ( size, -size,  size), ( size, -size, -size), ( size,  size, -size),
        ( size, -size,  size), ( size,  size, -size), ( size,  size,  size),

        # trás
        ( size, -size, -size), (-size, -size, -size), (-size,  size, -size),
        ( size, -size, -size), (-size,  size, -size), ( size,  size, -size),

        # esquerda
        (-size, -size, -size), (-size, -size,  size), (-size,  size,  size),
        (-size, -size, -size), (-size,  size,  size), (-size,  size, -size),

        # baixo
        (-size, -size, -size), ( size, -size, -size), ( size, -size,  size),
        (-size, -size, -size), ( size, -size,  size), (-size, -size,  size),

        # cima
        (-size,  size,  size), ( size,  size,  size), ( size,  size, -size),
        (-size,  size,  size), ( size,  size, -size), (-size,  size, -size),
    ]

    # UV padrão (cada face usa imagem inteira)
    uv = [
        (0,0), (1,0), (1,1),
        (0,0), (1,1), (0,1),
    ] * 6

    inicio = len(vertices_list)

    for v in cube:
        vertices_list.append(v)

    for t in uv:
        textures_coord_list.append(t)

    quantidade = len(cube)

    # carregar texturas (6 imagens)
    global numberTextures
    textura_ids = []

    for tex in texturas:
        load_texture_from_file_skybox(numberTextures, tex)
        textura_ids.append(numberTextures)
        numberTextures += 1

    return inicio, quantidade, textura_ids

def desenha_skybox(inicio, textura_ids):
    glDisable(GL_CULL_FACE) # ver por dentro
    
    glUniform1f(glGetUniformLocation(program, "matAmbient"), 0.3)
    glUniform1f(glGetUniformLocation(program, "matDiffuse"), 0.0)
    glUniform1f(glGetUniformLocation(program, "matSpecular"), 0.0)
    glUniform1f(glGetUniformLocation(program, "matShininess"), 1.0)

    for i in range(6):
        glBindTexture(GL_TEXTURE_2D, textura_ids[i])
        glDrawArrays(GL_TRIANGLES, inicio + i*6, 6)

    glEnable(GL_CULL_FACE)


skybox_texturas = [
    "objetos/skybox/cloudy/bluecloud_rt.jpg", # trás
    "objetos/skybox/cloudy/bluecloud_ft.jpg", # direita
    "objetos/skybox/cloudy/bluecloud_lf.jpg", # frente
    "objetos/skybox/cloudy/bluecloud_bk.jpg", # esquerda
    "objetos/skybox/cloudy/bluecloud_dn.jpg", # baixo
    "objetos/skybox/cloudy/bluecloud_up.jpg" # cima
    ]

skybox_inicio, skybox_qtd, skybox_tex = cria_skybox(skybox_texturas)

10
11
12
13
14
15


In [385]:
# =========================================================================
# 1. PISO (Interno da Garagem)
# =========================================================================
def cria_piso(textura, repeat=1):
    global vertices_list, textures_coord_list, normals_list

    size = 1.0  # tamanho grande

    # Vértices (2 triângulos)
    piso = [
        (-size,-0.1,-size), ( size,-0.1, size), ( size,-0.1,-size),
        (-size,-0.1,-size), (-size,-0.1, size), ( size,-0.1, size),
    ]

    # UV padrão
    uv = [
        (0,0), (repeat,0), (repeat,repeat),
        (0,0), (repeat,repeat), (0,repeat),
    ]

    # A normal de um piso horizontal virado para cima é sempre (0, 1, 0)
    normais = [
        (0.0, 1.0, 0.0), (0.0, 1.0, 0.0), (0.0, 1.0, 0.0),
        (0.0, 1.0, 0.0), (0.0, 1.0, 0.0), (0.0, 1.0, 0.0),
    ]

    inicio = len(vertices_list)

    for v in piso: vertices_list.append(v)
    for t in uv: textures_coord_list.append(t)
    for n in normais: normals_list.append(n)

    quantidade = len(piso)

    global numberTextures
    load_texture_from_file(numberTextures, textura)
    textura_id = numberTextures
    numberTextures += 1

    return inicio, quantidade, textura_id

def desenha_piso(inicio, textura_id, tx, ty, tz, sx=1, sy=1, sz=1):
    mat_model = model(0, 0,0,1, tx, ty, tz, sx, sy, sz)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    # Parâmetros de iluminação alinhados como Float
    glUniform1f(glGetUniformLocation(program, "matAmbient"), 0.2)
    glUniform1f(glGetUniformLocation(program, "matDiffuse"), 0.5)
    glUniform1f(glGetUniformLocation(program, "matSpecular"), 0.1)  # Chão pouco reflexivo
    glUniform1f(glGetUniformLocation(program, "matShininess"), 8.0)

    glBindTexture(GL_TEXTURE_2D, textura_id)
    glDrawArrays(GL_TRIANGLES, inicio, 6)

piso_inicio, piso_qtd, piso_tex = cria_piso("objetos/solo/piso.jpg", repeat=4)

16


In [386]:
# =========================================================================
# 2. TERRA (Externo)
# =========================================================================
def cria_terra(textura, repeat=1):
    global vertices_list, textures_coord_list, normals_list

    size = 1.0

    terra = [
        (-size,-0.1,-size), ( size,-0.1, size), ( size,-0.1,-size),
        (-size,-0.1,-size), (-size,-0.1, size), ( size,-0.1, size),
    ]

    uv = [
        (0,0), (repeat,0), (repeat,repeat),
        (0,0), (repeat,repeat), (0,repeat),
    ]

    # Normal para cima
    normais = [
        (0.0, 1.0, 0.0), (0.0, 1.0, 0.0), (0.0, 1.0, 0.0),
        (0.0, 1.0, 0.0), (0.0, 1.0, 0.0), (0.0, 1.0, 0.0),
    ]

    inicio = len(vertices_list)

    for v in terra: vertices_list.append(v)
    for t in uv: textures_coord_list.append(t)
    for n in normais: normals_list.append(n)

    quantidade = len(terra)

    global numberTextures
    load_texture_from_file(numberTextures, textura)
    textura_id = numberTextures
    numberTextures += 1

    return inicio, quantidade, textura_id

def desenha_terra(inicio, textura_id, tx, ty, tz, sx=1, sy=1, sz=1):
    mat_model = model(0, 0,0,1, tx, ty, tz, sx, sy, sz)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glUniform1f(glGetUniformLocation(program, "matAmbient"), 0.2)
    glUniform1f(glGetUniformLocation(program, "matDiffuse"), 0.6)
    glUniform1f(glGetUniformLocation(program, "matSpecular"), 0.0)  # Terra não tem brilho especular (opaca)
    glUniform1f(glGetUniformLocation(program, "matShininess"), 1.0)

    glBindTexture(GL_TEXTURE_2D, textura_id)
    glDrawArrays(GL_TRIANGLES, inicio, 6)

terra_inicio, terra_qtd, terra_tex = cria_terra("objetos/solo/terra.jpg", repeat=4)

17


In [387]:
# =========================================================================
# 3. GRAMA (Externo)
# =========================================================================
def cria_grama(blocos_x=20, blocos_z=20, tamanho_bloco=10.0):
    global vertices_list, textures_coord_list, normals_list

    inicio = len(vertices_list)
    
    # Calcula o deslocamento para centralizar a grade inteira no cenário (0,0)
    offset_x = (blocos_x * tamanho_bloco) / 2.0
    offset_z = (blocos_z * tamanho_bloco) / 2.0

    # Laço duplo para criar pequenos quadrados individuais de grama lado a lado
    for i in range(blocos_x):
        for j in range(blocos_z):
            # Coordenadas X e Z do bloco atual na grade
            x1 = (i * tamanho_bloco) - offset_x
            x2 = ((i + 1) * tamanho_bloco) - offset_x
            z1 = (j * tamanho_bloco) - offset_z
            z2 = ((j + 1) * tamanho_bloco) - offset_z
            
            y = -0.1  # Altura nativa do plano (será ajustada suavemente via ty no loop)

            # Triângulo 1 e Triângulo 2 que formam o quadrado deste bloco
            vertices_block = [
                (x1, y, z1), (x2, y, z2), (x2, y, z1),
                (x1, y, z1), (x1, y, z2), (x2, y, z2)
            ]

            # Coordenadas de textura repetindo de 0 a 1 em cada bloco individual
            uv_block = [
                (0.0, 0.0), (1.0, 1.0), (1.0, 0.0),
                (0.0, 0.0), (0.0, 1.0), (1.0, 1.0)
            ]

            # Normais perfeitas apontando rigidamente para CIMA (0, 1, 0)
            normais_block = [
                (0.0, 1.0, 0.0), (0.0, 1.0, 0.0), (0.0, 1.0, 0.0),
                (0.0, 1.0, 0.0), (0.0, 1.0, 0.0), (0.0, 1.0, 0.0)
            ]

            # Extende as listas globais sequencialmente
            for v in vertices_block: vertices_list.append(v)
            for t in uv_block: textures_coord_list.append(t)
            for n in normais_block: normals_list.append(n)

    # A quantidade total de vértices inseridos é o número de blocos multiplicado por 6
    quantidade = blocos_x * blocos_z * 6

    global numberTextures
    load_texture_from_file(numberTextures, "objetos/solo/grama.jpg")
    textura_id = numberTextures
    numberTextures += 1

    return inicio, quantidade, textura_id

def desenha_grama(inicio, quantidade, textura_id, tx, ty, tz, sx=1.0, sy=1.0, sz=1.0):
    mat_model = model(0, 0, 0, 1, tx, ty, tz, sx, sy, sz)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glUniform1f(glGetUniformLocation(program, "matAmbient"), 0.3)
    glUniform1f(glGetUniformLocation(program, "matDiffuse"), 0.6)
    glUniform1f(glGetUniformLocation(program, "matSpecular"), 0.02)
    glUniform1f(glGetUniformLocation(program, "matShininess"), 2.0)

    glBindTexture(GL_TEXTURE_2D, textura_id)
    
    # IMPORTANTE: Desenhamos usando a 'quantidade' calculada dinamicamente pela grade
    glDrawArrays(GL_TRIANGLES, inicio, quantidade)

# Inicialização da grama em grade (Removido o parâmetro repeat antigo)
grama_inicio, grama_qtd, grama_tex = cria_grama(blocos_x=20, blocos_z=20, tamanho_bloco=10.0)

18


In [388]:
# =========================================================================
# 4. CUBO (Usado para as Paredes e Teto da Garagem - Internos)
# =========================================================================
def cria_cubo(textura):
    global vertices_list, textures_coord_list, normals_list

    size = 1.0

    cube = [
        # frente
        (-size,-size, size), ( size,-size, size), ( size, size, size),
        (-size,-size, size), ( size, size, size), (-size, size, size),
        # trás
        (-size,-size,-size), (-size, size,-size), ( size, size,-size),
        (-size,-size,-size), ( size, size,-size), ( size,-size,-size),
        # esquerda
        (-size,-size,-size), (-size,-size, size), (-size, size, size),
        (-size,-size,-size), (-size, size, size), (-size, size,-size),
        # direita
        ( size,-size,-size), ( size, size,-size), ( size, size, size),
        ( size,-size,-size), ( size, size, size), ( size,-size, size),
        # topo
        (-size, size,-size), (-size, size, size), ( size, size, size),
        (-size, size,-size), ( size, size, size), ( size, size,-size),
        # base
        (-size,-size,-size), ( size,-size,-size), ( size,-size, size),
        (-size,-size,-size), ( size,-size, size), (-size,-size, size),
    ]

    repeat = 2 

    uv = [
        (0,0), (repeat,0), (repeat,repeat),
        (0,0), (repeat,repeat), (0,repeat),
    ] * 6

    # Vetores normais perpendiculares a cada uma das 6 faces do cubo
    normais = [
        # Frente (Z+)
        (0.0, 0.0, 1.0), (0.0, 0.0, 1.0), (0.0, 0.0, 1.0), (0.0, 0.0, 1.0), (0.0, 0.0, 1.0), (0.0, 0.0, 1.0),
        # Trás (Z-)
        (0.0, 0.0, -1.0), (0.0, 0.0, -1.0), (0.0, 0.0, -1.0), (0.0, 0.0, -1.0), (0.0, 0.0, -1.0), (0.0, 0.0, -1.0),
        # Esquerda (X-)
        (-1.0, 0.0, 0.0), (-1.0, 0.0, 0.0), (-1.0, 0.0, 0.0), (-1.0, 0.0, 0.0), (-1.0, 0.0, 0.0), (-1.0, 0.0, 0.0),
        # Direita (X+)
        (1.0, 0.0, 0.0), (1.0, 0.0, 0.0), (1.0, 0.0, 0.0), (1.0, 0.0, 0.0), (1.0, 0.0, 0.0), (1.0, 0.0, 0.0),
        # Topo (Y+)
        (0.0, 1.0, 0.0), (0.0, 1.0, 0.0), (0.0, 1.0, 0.0), (0.0, 1.0, 0.0), (0.0, 1.0, 0.0), (0.0, 1.0, 0.0),
        # Base (Y-)
        (0.0, -1.0, 0.0), (0.0, -1.0, 0.0), (0.0, -1.0, 0.0), (0.0, -1.0, 0.0), (0.0, -1.0, 0.0), (0.0, -1.0, 0.0)
    ]

    inicio = len(vertices_list)

    for v in cube: vertices_list.append(v)
    for t in uv: textures_coord_list.append(t)
    for n in normais: normals_list.append(n)

    quantidade = len(cube)

    global numberTextures
    load_texture_from_file(numberTextures, textura)
    textura_id = numberTextures
    numberTextures += 1

    return inicio, quantidade, textura_id

def desenha_cubo(inicio, textura_id, tx, ty, tz, ang, rx, ry, rz, sx, sy, sz):
    mat_model = model(ang, rx, ry, rz, tx, ty, tz, sx, sy, sz)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    # Coeficientes para o material de concreto/alvenaria das paredes
    glUniform1f(glGetUniformLocation(program, "matAmbient"), 0.2)
    glUniform1f(glGetUniformLocation(program, "matDiffuse"), 0.6)
    glUniform1f(glGetUniformLocation(program, "matSpecular"), 0.1)
    glUniform1f(glGetUniformLocation(program, "matShininess"), 16.0)

    glBindTexture(GL_TEXTURE_2D, textura_id)
    glDrawArrays(GL_TRIANGLES, inicio, 36)

cubo_inicio, cubo_qtd, cubo_tex = cria_cubo("objetos/parede/concreto.jpg")

19


In [389]:
# Parâmetros base da garagem (usados para posicionar paredes, teto e piso de forma consistente)

GARAGEM_CHAO = -1.1
GARAGEM_LARGURA = 2.0
GARAGEM_PROFUNDIDADE = 2.25
GARAGEM_ALTURA = 1.0

ESPESSURA_PAREDE = 0.10
OFFSET_INTERNO = 0.12

In [390]:
def desenha_garagem_externa():

    chao = GARAGEM_CHAO
    largura = GARAGEM_LARGURA
    profundidade = GARAGEM_PROFUNDIDADE
    altura = GARAGEM_ALTURA

    # Parede fundo externa
    desenha_cubo(
        cubo_inicio, cubo_tex,
        0,
        chao + altura,
        -profundidade,
        0,0,1,0,
        largura,
        altura,
        ESPESSURA_PAREDE
    )

    # Parede esquerda externa
    desenha_cubo(
        cubo_inicio, cubo_tex,
        -largura,
        chao + altura,
        0,
        0,0,1,0,
        ESPESSURA_PAREDE,
        altura,
        profundidade
    )

    # Parede direita externa
    desenha_cubo(
        cubo_inicio, cubo_tex,
        largura,
        chao + altura,
        0,
        0,0,1,0,
        ESPESSURA_PAREDE,
        altura,
        profundidade
    )

    # Teto externo
    desenha_cubo(
        cubo_inicio, cubo_tex,
        0,
        chao + 2*altura,
        0,
        0,0,1,0,
        largura,
        ESPESSURA_PAREDE,
        profundidade
    )

In [391]:
def desenha_garagem_interna():

    chao = GARAGEM_CHAO
    largura = GARAGEM_LARGURA
    profundidade = GARAGEM_PROFUNDIDADE
    altura = GARAGEM_ALTURA

    off = OFFSET_INTERNO

    # Parede fundo interna
    desenha_cubo(
        cubo_inicio, cubo_tex,
        0,
        chao + altura,
        -profundidade + off,
        0,0,1,0,
        largura - off,
        altura,
        0.05
    )

    # Parede esquerda interna
    desenha_cubo(
        cubo_inicio, cubo_tex,
        -largura + off,
        chao + altura,
        0,
        0,0,1,0,
        0.05,
        altura,
        profundidade - off
    )

    # Parede direita interna
    desenha_cubo(
        cubo_inicio, cubo_tex,
        largura - off,
        chao + altura,
        0,
        0,0,1,0,
        0.05,
        altura,
        profundidade - off
    )

    # Teto interno
    desenha_cubo(
        cubo_inicio, cubo_tex,
        0,
        chao + 2*altura - off,
        0,
        0,0,1,0,
        largura - off,
        0.05,
        profundidade - off
    )

In [392]:
# =========================================================================
# 5. PORTÃO DA GARAGEM
# =========================================================================
cubo_porta_inicio, cubo_porta_qtd, cubo_porta_tex = cria_cubo("objetos/porta/porta.jpg")

def desenha_porta():
    chao = -0.6
    largura = 1.9
    altura = 1.0
    espessura = 0.05
    frente = 2.3

    mat_model = model(0, 0,1,0, 0, chao + altura/2 + porta_offset, frente, largura, altura, espessura)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    # Parâmetros customizados para o portão
    glUniform1f(glGetUniformLocation(program, "matAmbient"), 0.25)
    glUniform1f(glGetUniformLocation(program, "matDiffuse"), 0.7)
    glUniform1f(glGetUniformLocation(program, "matSpecular"), 0.4) # Mais reflexivo que as paredes
    glUniform1f(glGetUniformLocation(program, "matShininess"), 32.0)

    glBindTexture(GL_TEXTURE_2D, cubo_porta_tex)
    glDrawArrays(GL_TRIANGLES, cubo_porta_inicio, 36)

20


### Requisitando buffer

In [393]:
buffer_VBO = glGenBuffers(3)

### Enviando vértices para a GPU

In [394]:
vertices = np.zeros(len(vertices_list), [("position", np.float32, 3)])
vertices['position'] = vertices_list


# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_STATIC_DRAW)
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)
loc_vertices = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc_vertices)
glVertexAttribPointer(loc_vertices, 3, GL_FLOAT, False, stride, offset)

### Enviando textura para GPU

In [395]:
textures = np.zeros(len(textures_coord_list), [("position", np.float32, 2)]) # duas coordenadas
textures['position'] = textures_coord_list


# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[1])
glBufferData(GL_ARRAY_BUFFER, textures.nbytes, textures, GL_STATIC_DRAW)
stride = textures.strides[0]
offset = ctypes.c_void_p(0)
loc_texture_coord = glGetAttribLocation(program, "texture_coord")

glEnableVertexAttribArray(loc_texture_coord)
glVertexAttribPointer(loc_texture_coord, 2, GL_FLOAT, False, stride, offset)

### Enviando normais para a GPU

In [396]:
normals = np.zeros(len(normals_list),[("position", np.float32, 3)])
normals["position"] = normals_list

# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[2])
glBufferData(GL_ARRAY_BUFFER, normals.nbytes, normals, GL_STATIC_DRAW)

loc_normal = glGetAttribLocation(program, "normal")

glEnableVertexAttribArray(loc_normal)
glVertexAttribPointer(loc_normal,3, GL_FLOAT, False, normals.strides[0], ctypes.c_void_p(0))

### Estado inicial da cena

In [397]:
# =====================
# ESTADO INICIAL
# =====================
INIT_cameraPos = glm.vec3(0.0, 0.0, 4.0)
INIT_cameraFront = glm.vec3(0.0, 0.0, -1.0)
INIT_yaw = -90.0
INIT_pitch = 0.0

INIT_car_x = 0.0
INIT_car_z = 0.0
INIT_car_angle = 0.0

INIT_pneu_scale = 0.05
INIT_porta_offset = 2.0

### Função de hitbox do carro

In [398]:
def carro_em_area_valida(x, z):

    # =====================
    # GARAGEM 
    # =====================
    dentro_garagem = (-0.65 <= x <= 0.65 and -0.8 <= z <= 3.5)

    if dentro_garagem:
        # só entra se a porta estiver aberta
        if porta_offset >= 1.25:
            return True
        else:
            return False

    # =====================
    # ESTRADA
    # =====================
    if -100.0 <= x <= 12.0 and 10.0 <= z <= 14:
        return True

    # =====================
    # TERRA 
    # =====================
    if -1.0 <= x <= 1.0 and 3.5 <= z <= 10.0:
        return True

    return False

def carro_debaixo_do_portao(x, z):
    return (-1.0 <= x <= 1.0) and (0.9 <= z <= 3.6)

### Funções do teclado

In [ ]:
def reset_scene():
    global cameraPos, cameraFront, yaw, pitch
    global car_x, car_z, car_angle
    global pneu_scale, porta_offset

    cameraPos = glm.vec3(INIT_cameraPos)
    cameraFront = glm.vec3(INIT_cameraFront)
    yaw = INIT_yaw
    pitch = INIT_pitch

    car_x = INIT_car_x
    car_z = INIT_car_z
    car_angle = INIT_car_angle

    pneu_scale = INIT_pneu_scale
    porta_offset = INIT_porta_offset


def dentro_dos_limites(pos):
    limite_x = 10.0
    z_min = -2.5
    z_max = 17.5
    min_y = -1.0   # não atravessa o chão
    max_y = 20.0    # altura máxima 

    return (
        -limite_x <= pos.x <= limite_x and
        z_min <= pos.z <= z_max and
        min_y <= pos.y <= max_y
    )

def key_callback(window, key, scancode, action, mods):
    global porta_offset, car_x, car_z, car_angle, pneu_scale
    global vol_dir, rot, vel
    global cameraPos, cameraFront, cameraUp, polygonal_mode, deltaTime
    
    global ambientStrength, diffuseStrength, specularStrength
    global ambient_on, lampada_on, farol_on, segunda_luz_on, poste_on

    if key == glfw.KEY_ESCAPE and action == glfw.PRESS:
        glfw.set_window_should_close(window, True)

    if action == glfw.PRESS or action == glfw.REPEAT:

        # =====================
        # PORTA
        # =====================
        if key == glfw.KEY_O:
            porta_offset = min(porta_offset + 0.05, 2.0)

        if key == glfw.KEY_L:
            if porta_offset > 0.0:
                if not carro_debaixo_do_portao(car_x, car_z):
                    porta_offset = max(porta_offset - 0.05, 0.0)

        # =====================
        # CARRO
        # =====================
        if key == glfw.KEY_UP:
            novo_angle = car_angle + vol_dir * rot
            novo_x = car_x + vel * math.sin(math.radians(novo_angle))
            novo_z = car_z + vel * math.cos(math.radians(novo_angle))
            if carro_em_area_valida(novo_x, novo_z):
                car_angle = novo_angle; car_x = novo_x; car_z = novo_z

        if key == glfw.KEY_DOWN:
            novo_angle = car_angle - vol_dir * rot
            novo_x = car_x - vel * math.sin(math.radians(novo_angle))
            novo_z = car_z - vel * math.cos(math.radians(novo_angle))
            if carro_em_area_valida(novo_x, novo_z):
                car_angle = novo_angle; car_x = novo_x; car_z = novo_z

        if key == glfw.KEY_LEFT:
            vol_dir = +1

        if key == glfw.KEY_RIGHT:
            vol_dir = -1

        if key == glfw.KEY_SPACE:
            vol_dir = 0

        # =====================
        # PNEU
        # =====================
        if key == glfw.KEY_I:
            pneu_scale = min(pneu_scale + 0.05, 0.45)

        if key == glfw.KEY_K:
            pneu_scale = max(0.05, pneu_scale - 0.05)

        # ==============================
        # ILUMINAÇÃO
        # ==============================

        # Interruptores independentes (Requisito 3)

        # Controle independente das fontes de luz:
        # 1 - luz ambiente
        # 2 - lâmpada do teto
        # 3 - faróis do carro
        # 4 - lanterna
        # 5 - poste
        
        if key == glfw.KEY_1 and action == glfw.PRESS:
            ambient_on = not ambient_on

        if key == glfw.KEY_2 and action == glfw.PRESS:
            lampada_on = not lampada_on

        if key == glfw.KEY_3 and action == glfw.PRESS:
            farol_on = not farol_on

        if key == glfw.KEY_4 and action == glfw.PRESS:
            segunda_luz_on = not segunda_luz_on

        if key == glfw.KEY_5 and action == glfw.PRESS:
            poste_on = not poste_on

        # Incrementar e Decrementar Intensidades Globais (Requisitos 4, 5 e 6)
        
        # Intensidade global da componente ambiente
        if key == glfw.KEY_Z:
            ambientStrength = max(0.0, ambientStrength - 0.05)
        if key == glfw.KEY_X:
            ambientStrength = min(1.0, ambientStrength + 0.05)

        # Intensidade global da reflexão difusa.
        if key == glfw.KEY_C:
            diffuseStrength = max(0.0, diffuseStrength - 0.05)
        if key == glfw.KEY_V:
            diffuseStrength = min(2.0, diffuseStrength + 0.05)

        # Intensidade global da reflexão especular.
        if key == glfw.KEY_B:
            specularStrength = max(0.0, specularStrength - 0.05)
        if key == glfw.KEY_N:
            specularStrength = min(2.0, specularStrength + 0.05)

        # =====================
        # CÂMERA (WASD)
        # =====================
        cameraSpeed = 10.0 * deltaTime

        if key == glfw.KEY_W:
            nova_pos = cameraPos + cameraSpeed * cameraFront
            if dentro_dos_limites(nova_pos): cameraPos = nova_pos

        if key == glfw.KEY_S:
            nova_pos = cameraPos - cameraSpeed * cameraFront
            if dentro_dos_limites(nova_pos): cameraPos = nova_pos

        if key == glfw.KEY_A:
            direita = glm.normalize(glm.cross(cameraFront, cameraUp))
            nova_pos = cameraPos - direita * cameraSpeed
            if dentro_dos_limites(nova_pos): cameraPos = nova_pos

        if key == glfw.KEY_D:
            direita = glm.normalize(glm.cross(cameraFront, cameraUp))
            nova_pos = cameraPos + direita * cameraSpeed
            if dentro_dos_limites(nova_pos): cameraPos = nova_pos

        if key == glfw.KEY_R and action == glfw.PRESS:
            reset_scene()
            vol_dir = 0

        if key == glfw.KEY_P and action == glfw.PRESS:
            polygonal_mode = not polygonal_mode

### Eventos da câmera

In [400]:
# camera
cameraPos   = glm.vec3(INIT_cameraPos)
cameraFront = glm.vec3(INIT_cameraFront)
cameraUp    = glm.vec3(0.0, 1.0, 0.0)

firstMouse = True
yaw   = INIT_yaw	# yaw is initialized to -90.0 degrees since a yaw of 0.0 results in a direction vector pointing to the right so we initially rotate a bit to the left.
pitch =  INIT_pitch
lastX =  largura / 2.0
lastY =  altura / 2.0
fov   =  45.0

# timing
deltaTime = 0.0	# time between current frame and last frame
lastFrame = 0.0


firstMouse = True
yaw = INIT_yaw
pitch = INIT_pitch
lastX =  largura/2
lastY =  altura/2

def framebuffer_size_callback(window, largura, altura):

    # make sure the viewport matches the new window dimensions note that width and 
    # height will be significantly larger than specified on retina displays.
    glViewport(0, 0, largura, altura)

# glfw: whenever the mouse moves, this callback is called
# -------------------------------------------------------
def mouse_callback(window, xpos, ypos):
    global cameraFront, lastX, lastY, firstMouse, yaw, pitch
    
    if (firstMouse):

        lastX = xpos
        lastY = ypos
        firstMouse = False

    xoffset = xpos - lastX
    yoffset = lastY - ypos # reversed since y-coordinates go from bottom to top
    lastX = xpos
    lastY = ypos

    sensitivity = 0.1 # change this value to your liking
    xoffset *= sensitivity
    yoffset *= sensitivity

    yaw += xoffset
    pitch += yoffset

    # make sure that when pitch is out of bounds, screen doesn't get flipped
    if (pitch > 89.0):
        pitch = 89.0
    if (pitch < -89.0):
        pitch = -89.0

    front = glm.vec3()
    front.x = glm.cos(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    front.y = glm.sin(glm.radians(pitch))
    front.z = glm.sin(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    cameraFront = glm.normalize(front)

# glfw: whenever the mouse scroll wheel scrolls, this callback is called
# ----------------------------------------------------------------------
def scroll_callback(window, xoffset, yoffset):
    global fov

    fov -= yoffset
    if (fov < 1.0):
            fov = 1.0
    if (fov > 45.0):
            fov = 45.0
        
glfw.set_key_callback(window,key_callback)
glfw.set_framebuffer_size_callback(window, framebuffer_size_callback)
glfw.set_cursor_pos_callback(window, mouse_callback)
glfw.set_scroll_callback(window, scroll_callback)

# tell GLFW to capture our mouse
glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)

### Matriz model, view e projection

In [401]:
def model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    
    angle = math.radians(angle)
    
    matrix_transform = glm.mat4(1.0) # instanciando uma matriz identidade
       
    # aplicando translacao (terceira operação a ser executada)
    matrix_transform = glm.translate(matrix_transform, glm.vec3(t_x, t_y, t_z))    
    
    # aplicando rotacao (segunda operação a ser executada)
    if angle!=0:
        matrix_transform = glm.rotate(matrix_transform, angle, glm.vec3(r_x, r_y, r_z))
    
    # aplicando escala (primeira operação a ser executada)
    matrix_transform = glm.scale(matrix_transform, glm.vec3(s_x, s_y, s_z))
    
    matrix_transform = np.array(matrix_transform)
    
    return matrix_transform

def view():
    global cameraPos, cameraFront, cameraUp
    mat_view = glm.lookAt(cameraPos, cameraPos + cameraFront, cameraUp);
    mat_view = np.array(mat_view)
    return mat_view

def projection():
    global altura, largura
    # perspective parameters: fovy, aspect, near, far
    mat_projection = glm.perspective(glm.radians(fov), largura/altura, 0.1, 1000.0)

    
    mat_projection = np.array(mat_projection)    
    return mat_projection

def view_skybox():
    global cameraPos, cameraFront, cameraUp

    mat_view = glm.lookAt(cameraPos, cameraPos + cameraFront, cameraUp)

    # remove translação
    mat_view = glm.mat4(glm.mat3(mat_view))

    return np.array(mat_view)

### Exibindo janela

In [402]:
glfw.show_window(window)
glfw.set_key_callback(window, key_callback)

<function __main__.key_callback(window, key, scancode, action, mods)>

### Loop principal

In [403]:
glEnable(GL_DEPTH_TEST)   # importante para 3D
polygonal_mode = False

# =========================
# ILUMINAÇÃO
# =========================

ambientStrength = 0.75
diffuseStrength = 1.0
specularStrength = 0.5

ambient_on = True

lampada_on = True
farol_on = True

segunda_luz_on = True

poste_on = True


#=================================
# VARIÁVEIS GLOBAIS PARA ANIMAÇÃO
#=================================

# variáveis para animação
porta_offset = 2.0  # controla abertura (0 = fechada) (2.0 = totalmente aberta)

car_x = INIT_car_x
car_z = INIT_car_z
car_angle = INIT_car_angle

vol_dir = 0   # 1 = esquerda, 0 = reto, -1 = direita
vel = 0.1
rot = 2.0  # quanto gira por frame


pneu_scale = INIT_pneu_scale  # controla o estado do pneu (0 = murcho, 1 = cheio)
scale_full = INIT_porta_offset  # valor de pneu_scale para o pneu estar completamente cheio (ajuste para que fique visualmente adequado)

while not glfw.window_should_close(window):

    currentFrame = glfw.get_time()
    deltaTime = currentFrame - lastFrame
    lastFrame = currentFrame

    glfw.poll_events()

    glClearColor(0.05, 0.05, 0.1, 1.0)
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

    if polygonal_mode:
        glPolygonMode(GL_FRONT_AND_BACK, GL_LINE)
    else:
        glPolygonMode(GL_FRONT_AND_BACK, GL_FILL)

    glUseProgram(program)

    # Ativação dos modificadores de força globais via teclado
    glUniform1f(glGetUniformLocation(program, "globalAmbientStrength"), ambientStrength if ambient_on else 0.0)
    glUniform1f(glGetUniformLocation(program, "globalDiffuseStrength"), diffuseStrength)
    glUniform1f(glGetUniformLocation(program, "globalSpecularStrength"), specularStrength)
    
    glUniform1i(
    glGetUniformLocation(program, "postesOn"),
    int(poste_on)
)

    # Mapeamento dos locais dos Uniforms de Materiais e Ambiente
    loc_ambiente_id = glGetUniformLocation(program, "ambienteID")
    loc_mat_amb     = glGetUniformLocation(program, "matAmbient")
    loc_mat_diff    = glGetUniformLocation(program, "matDiffuse")
    loc_mat_spec    = glGetUniformLocation(program, "matSpecular")
    loc_mat_shin    = glGetUniformLocation(program, "matShininess")

    # ==========================================================
    # PROJECTION & VIEW
    # ==========================================================
    mat_projection = projection()
    glUniformMatrix4fv(glGetUniformLocation(program, "projection"), 1, GL_TRUE, mat_projection)

    # Configuração das matrizes do Skybox
    mat_view = view_skybox()
    glUniformMatrix4fv(glGetUniformLocation(program, "view"), 1, GL_TRUE, mat_view)
    mat_model = np.identity(4)
    glUniformMatrix4fv(glGetUniformLocation(program, "model"), 1, GL_TRUE, mat_model)

    glDepthFunc(GL_LEQUAL)
    glDepthMask(GL_FALSE)
    desenha_skybox(skybox_inicio, skybox_tex)
    glDepthMask(GL_TRUE)
    glDepthFunc(GL_LESS)
            
    mat_view = view()
    glUniformMatrix4fv(glGetUniformLocation(program, "view"), 1, GL_TRUE, mat_view)

    glUniform3f(glGetUniformLocation(program, "viewPos"), cameraPos.x, cameraPos.y, cameraPos.z)

    # ==========================================================
    # ATUALIZAÇÃO E ENVIO DOS DADOS DAS FONTES DE LUZ
    # ==========================================================

    # 1. Luzes Internas (Lâmpada do Teto e Segunda Luz)
    
    glUniform3f(glGetUniformLocation(program, "lampadaTetoPos"),0.0, 1.2, 0.0)
    glUniform3f(glGetUniformLocation(program, "lampadaTetoDir"),0.0, -1.0, 0.0)
    glUniform1f(glGetUniformLocation(program, "cutOffInternoLampada"),math.cos(math.radians(30.0)))
    glUniform1f(glGetUniformLocation(program, "outerCutOffInternoLampada"),math.cos(math.radians(45.0)))
    cor_teto = (1.5, 1.5, 1.2) if lampada_on else (0.0, 0.0, 0.0) 
    glUniform3f(glGetUniformLocation(program, "lampadaTetoColor"), *cor_teto)

    # Segunda luz interna (lanterna)
    glUniform3f(glGetUniformLocation(program, "segundaLuzPos"),1.95,-0.95,2.15)
    glUniform3f(glGetUniformLocation(program, "segundaLuzDir"),-0.7,0.0,-0.7)
    glUniform1f(glGetUniformLocation(program, "cutOffInternoLanterna"),math.cos(math.radians(10.0)))
    glUniform1f(glGetUniformLocation(program, "outerCutOffInternoLanterna"),math.cos(math.radians(15.0)))
    cor_segunda = (0.0, 1.3, 1.6) if segunda_luz_on else (0.0, 0.0, 0.0) 
    glUniform3f(glGetUniformLocation(program, "segundaLuzColor"), *cor_segunda)

    # 2. Luzes Externas (Dois Faróis calculados via trigonometria a partir da frente do veículo)
    rad = math.radians(car_angle)
    frente_x = math.sin(rad)
    frente_z = math.cos(rad)
    direita_x = math.cos(rad)
    direita_z = -math.sin(rad)

    dist_frente = 1.2
    dist_lateral = 0.35
    farol_y = 0.0

    farolEsq_x = car_x + (frente_x * dist_frente) - (direita_x * dist_lateral)
    farolEsq_z = car_z + (frente_z * dist_frente) - (direita_z * dist_lateral)

    farolDir_x = car_x + (frente_x * dist_frente) + (direita_x * dist_lateral)
    farolDir_z = car_z + (frente_z * dist_frente) + (direita_z * dist_lateral)

    glUniform3f(glGetUniformLocation(program, "farolEsqPos"), farolEsq_x, farol_y, farolEsq_z)
    glUniform3f(glGetUniformLocation(program, "farolDirPos"), farolDir_x, farol_y, farolDir_z)
    glUniform3f(glGetUniformLocation(program, "farolEsqDir"), frente_x, -0.4, frente_z)
    glUniform3f(glGetUniformLocation(program, "farolDirDir"), frente_x, -0.4, frente_z)
    glUniform1f(glGetUniformLocation(program, "cutOff"), math.cos(math.radians(18.0)))
    glUniform1f(glGetUniformLocation(program, "outerCutOff"), math.cos(math.radians(25.0)))
    cor_farol = (1.2, 1.2, 1.0) if farol_on else (0.0, 0.0, 0.0)
    glUniform3f(glGetUniformLocation(program, "farolColor"), * cor_farol)

    # ==========================================================
    # DESENHO DOS OBJETOS INTERNOS (ambienteID = 1)
    # ==========================================================
    glUniform1i(loc_ambiente_id, 1)

    # Garagem: Opaca, pouco reflexo especular
    glUniform1f(loc_mat_amb, 0.3); glUniform1f(loc_mat_diff, 0.6); glUniform1f(loc_mat_spec, 0.1); glUniform1f(loc_mat_shin, 8.0)
    desenha_garagem_interna()
    desenha_piso(piso_inicio, piso_tex, 0, -0.99, 0, 2.0, 1, 2.25)

    # Pneu: Borracha fosca
    glUniform1f(loc_mat_amb, 0.2); glUniform1f(loc_mat_diff, 0.5); glUniform1f(loc_mat_spec, 0.05); glUniform1f(loc_mat_shin, 4.0)
    t = max(0.0, min(1.0, pneu_scale / scale_full))
    scale_y = 0.30 + 0.15 * t
    scale_xz = 0.50 - 0.05 * t
    pneu_y = -1.13 + (0.30 / 2.0) + (scale_y - 0.30) * 0.4
    desenha_pneu(90, 0, 1, 0, -1.8, pneu_y, 0.5, scale_xz, scale_y, scale_xz, 3)

    # Ferramentas: Metálicas brilhantes
    glUniform1f(loc_mat_amb, 0.4); glUniform1f(loc_mat_diff, 0.8); glUniform1f(loc_mat_spec, 1.0); glUniform1f(loc_mat_shin, 128.0)
    desenha_ferramentas(0.0, 0, 1, 0, -1.2, -1.09, -1.95, 0.1, 0.1, 0.1, 5)

    # Gasolina: Plástico brilhante
    glUniform1f(loc_mat_amb, 0.5); glUniform1f(loc_mat_diff, 0.7); glUniform1f(loc_mat_spec, 0.4); glUniform1f(loc_mat_shin, 32.0)
    desenha_gasolina(180, 0, 1, 0, 2.0, -0.95, -2.1, 0.08, 0.08, 0.08, 6)

    # Lâmpada do Teto: Superfície polida / reflexiva
    # Lanterna: Plástico brilhante
    glUniform1f(loc_mat_amb, 0.4); glUniform1f(loc_mat_diff, 0.8); glUniform1f(loc_mat_spec, 0.5); glUniform1f(loc_mat_shin, 16.0)
    desenha_lampada(0.0, 0, 1, 0, 0, 0.23, 0, 0.5, 0.5, 0.5, 7)
    desenha_lanterna(225, 0, 1, 0, 1.7, -1.08, 1.9, 1.0, 1.0, 1.0, 9)

    # ==========================================================
    # DESENHO DOS OBJETOS EXTERNOS (ambienteID = 0)
    # ==========================================================
    glUniform1i(loc_ambiente_id, 0)

    desenha_garagem_externa()
    desenha_porta()

    # Solo, Grama e Terra: Praticamente sem brilho especular
    glUniform1f(loc_mat_amb, 0.3); glUniform1f(loc_mat_diff, 0.6); glUniform1f(loc_mat_spec, 0.0); glUniform1f(loc_mat_shin, 1.0)
    desenha_grama(grama_inicio, grama_qtd, grama_tex, 0, -1.02, 0, 1.0, 1.0, 1.0)
    for i in range(3):
        desenha_terra(terra_inicio, terra_tex, 0, -0.995, 3 + i * 4.0, 1.5, 1, 2.0)

    # Estrada (Asfalto fosco)
    glUniform1f(loc_mat_amb, 0.2); glUniform1f(loc_mat_diff, 0.5); glUniform1f(loc_mat_spec, 0.1); glUniform1f(loc_mat_shin, 2.0)
    for i in range(8):
        desenha_rua(90, 0, 1, 0, 0 + i * 13.15, -1.12, 12.5, 0.4, 0.4, 0.4, 0)
        desenha_rua(90, 0, 1, 0, 0 - i * 13.15, -1.12, 12.5, 0.4, 0.4, 0.4, 0)

    # Postes, Carros e Cones: Superfícies polidas / reflexivas
    glUniform1f(loc_mat_amb, 0.3); glUniform1f(loc_mat_diff, 0.7); glUniform1f(loc_mat_spec, 0.5); glUniform1f(loc_mat_shin, 32.0)
    posicoesPostesX = [-85.0, -55.0, -25.0, 5.0, 35.0, 65.0]
    for x_p in posicoesPostesX:
        desenha_poste(90, 0, 1, 0, x_p, -1.1, 15, 0.5, 0.5, 0.5, 1)

    # Carros (Pintura automotiva brilhante)
    glUniform1f(loc_mat_amb, 0.4); glUniform1f(loc_mat_diff, 0.8); glUniform1f(loc_mat_spec, 1.0); glUniform1f(loc_mat_shin, 90.0)
    desenha_carro(car_angle, 0, 1, 0, car_x, -0.75, car_z, 0.8, 0.8, 0.8, 2)
    desenha_carro(180, 0, 0, 1, 17.0, -0.70, 12.5, 0.8, 0.8, 0.8, 2)

    # Cones
    glUniform1f(loc_mat_amb, 0.3); glUniform1f(loc_mat_diff, 0.7); glUniform1f(loc_mat_spec, 0.5); glUniform1f(loc_mat_shin, 32.0)
    for i in range(5):
        desenha_cone(0.0, 0, 0, 1, 15.0, -1.0, 11.5 + i*0.5, 0.5, 0.5, 0.5, 4)

    # Árvores (Foscas)
    glUniform1f(loc_mat_amb, 0.2); glUniform1f(loc_mat_diff, 0.5); glUniform1f(loc_mat_spec, 0.0); glUniform1f(loc_mat_shin, 1.0)
    for i in range(20):
        desenha_arvore(0.0, 0, 0, 1, 0 + i * 5, -0.975, 17.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -5 - i * 5, -0.975, 17.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, 2.5 + i * 5, -0.975, 22.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -2.5 - i * 5, -0.975, 22.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, 0 + i * 5, -0.975, 27.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -5 - i * 5, -0.975, 27.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, 2.5 + i * 5, -0.975, 32.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -2.5 - i * 5, -0.975, 32.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, 5.0 + i * 5, -0.975, 7.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -5.0 - i * 5, -0.975, 7.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, 5 + i * 5, -0.975, 2.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -5 - i * 5, -0.975, 2.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, 5.0 + i * 5, -0.975, -2.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -5.0 - i * 5, -0.975, -2.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, 0 + i * 5, -0.975, -7.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -5 - i * 5, -0.975, -7.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -2.5 + i * 5, -0.975, -12.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -7.5 - i * 5, -0.975, -12.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, 0 + i * 5, -0.975, -17.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -5 - i * 5, -0.975, -17.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -2.5 + i * 5, -0.975, -22.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -7.5 - i * 5, -0.975, -22.5, 0.4, 0.4, 0.4, 8)

    glfw.swap_buffers(window)

glfw.terminate()